[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/08_multimodal_agents/08_multimodal_agents.ipynb)

# 08. Multimodal Agents

**This notebook covers:**
- Tool-use with vision models
- Visual grounding example
- Multi-step planning (ReAct-style)

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/05_Advanced_Topics/08_multimodal_agents"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Tool Registry for Vision Agents


In [ ]:
TOOLS = {
    "detect_objects": lambda img, label: [{"label": label, "bbox": [0.1, 0.2, 0.4, 0.5]}],
    "crop_region": lambda img, bbox: img,
    "ocr": lambda img: "STOP",
    "calculator": lambda expr: str(eval(expr)),
}

def call_tool(name, **kwargs):
    if name not in TOOLS:
        return f"Unknown tool: {name}"
    return TOOLS[name](**kwargs)

print("Available tools:", list(TOOLS.keys()))
print("OCR result:", call_tool("ocr", img=None))

## 2. Visual Grounding Example


In [ ]:
def ground_phrase(detections, phrase):
    phrase = phrase.lower()
    for det in detections:
        if det["label"] in phrase:
            return det["bbox"]
    return None

image = torch.rand(3, 224, 224)
dets = call_tool("detect_objects", img=image, label="cat")
bbox = ground_phrase(dets, "find the cat in the image")
print("Grounded bbox:", bbox)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image.permute(1, 2, 0).numpy())
if bbox:
    x1, y1, x2, y2 = bbox
    ax.add_patch(plt.Rectangle((x1*224, y1*224), (x2-x1)*224, (y2-y1)*224,
                              fill=False, edgecolor='lime', linewidth=2))
ax.set_title('Visual grounding'); ax.axis('off'); plt.show()

## 3. Multi-Step ReAct Planning


In [ ]:
class MultimodalAgent:
    def __init__(self, tools):
        self.tools = tools
        self.trace = []

    def plan(self, goal):
        if "sign" in goal.lower():
            return ["detect_objects(traffic_sign)", "crop_region(bbox)", "ocr(crop)"]
        if "count" in goal.lower():
            return ["detect_objects(object)", "calculator(count)"]
        return ["detect_objects(all)"]

    def run(self, goal, image):
        self.trace = []
        for action in self.plan(goal):
            self.trace.append(f"Thought: need {action}")
            if action.startswith("detect"):
                obs = self.tools["detect_objects"](image, "traffic_sign")
            elif action.startswith("ocr"):
                obs = self.tools["ocr"](image)
            else:
                obs = action
            self.trace.append(f"Action: {action}")
            self.trace.append(f"Observation: {obs}")
        return self.trace

agent = MultimodalAgent(TOOLS)
trace = agent.run("Read the text on the traffic sign", image)
for line in trace:
    print(line)

## 4. Agent Loop Visualization


In [ ]:
steps = [t for t in trace if t.startswith('Action')]
plt.figure(figsize=(8, 2))
plt.barh(range(len(steps)), [1]*len(steps), color='teal')
plt.yticks(range(len(steps)), steps)
plt.xlabel('Step'); plt.title('Multi-step agent plan'); plt.tight_layout(); plt.show()

## Summary

Built a tool-using multimodal agent with visual grounding and ReAct-style planning.

**Congratulations!** You completed Module 05 Advanced Topics.
